# XGBoost: a referência do grupo

O XGBoost é a linha `vs_xgboost` de todos os modelos no placar comum (`docs/pre-proposta.md`, "Placar comum").
Este notebook gera essa linha: `("xgboost", "final")` em `resultados/placar.jsonl`.

1. **Trial 0:** os padrões do XGBoost, no placar.
2. **Busca:** Optuna nas 5 dobras oficiais, com o procedimento do placar em cada dobra (`comum.estudo_placar`).
3. **Melhor trial no placar:** medido de novo nas 15 divisões do placar, que a busca não viu.
4. **Dados:** sem oversampling e com as três features do `docs/GUIA_DADOS.md`, contra o melhor trial.
5. **Final:** só troca o melhor trial por uma variante que **ganhou** dele. É a linha que os outros modelos enxergam.
6. **Teste:** uma vez, com `REVELAR_TESTE = True`, depois de tudo congelado.

Em cada dobra, 80% do treino ajustam e 20% decidem o número de árvores; depois o modelo retreina no treino inteiro da dobra com esse número.
A dobra de validação só mede.

O espaço de busca e as perdas estão em `modelo.py`, com a medida que justifica cada escolha.

## Como executar

Na CPU, localmente, a partir de qualquer pasta do repositório: `jupyter nbconvert --to notebook --execute --inplace experimentos/xgboost/xgboost.ipynb`.
O CSV é procurado em `CHURN_CSV`, em `dados/telco_ibm.csv`, no Input do Kaggle e, por fim, baixado pelo `kagglehub`.
O estudo fica em `resultados/xgboost/xgboost.db` e retoma de onde parou.

In [1]:
# ===== Configuração da execução (edite aqui) =====
N_TRIALS = 100          # mesmo N para todos os modelos (pré-proposta)
N_SEMENTES = 5          # sementes da avaliação final no teste
REVELAR_TESTE = False   # True só com a configuração congelada: o teste é usado uma única vez
MODO_TESTE = False      # True = passada rápida para validar o notebook, em uma pasta descartável

In [2]:
import json, os, sys, time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").exists())
sys.path[:0] = [str(REPO), str(REPO / "experimentos" / "xgboost")]
import fluxo

os.environ["CHURN_CSV"] = fluxo.achar_csv(os.environ.get("CHURN_CSV", ""))
if MODO_TESTE:
    os.environ["CHURN_RESULTADOS"] = str(REPO / "execucoes" / time.strftime("teste-xgboost-%Y%m%d-%H%M"))
    N_TRIALS, N_SEMENTES = 3, 2

import numpy as np, optuna, pandas as pd, xgboost as xgb
from scipy.stats import ks_2samp
import comum, modelo as M

PASTA = comum.RESULTADOS
PASTA_ESTUDO = os.path.join(PASTA, "xgboost")
os.makedirs(PASTA_ESTUDO, exist_ok=True)
pd.set_option("display.max_columns", None, "display.width", 200)
print("xgboost", xgb.__version__, "| optuna", optuna.__version__, "| CSV:", os.environ["CHURN_CSV"])
print("placar em", PASTA)

xgboost 3.4.1 | optuna 5.0.0 | CSV: C:\Users\rafae\.cache\kagglehub\datasets\kapturovalexander\customers-churned-in-telecom-services\versions\3\customer_churn_telecom_services.csv
placar em C:\dev\Faculdade\churn_project\resultados


In [3]:
# Verificações rápidas antes de gastar tempo de treino
X_tr, X_te, y_tr, y_te = comum.dividir(comum.carregar_df())
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis"
if MODO_TESTE:  # o teste de verdade nunca entra numa passada de conferência: um pedaço do treino faz o papel dele
    X_te, y_te = X_tr.iloc[:500], y_tr.iloc[:500]
ASS = comum.assinatura_treino(X_tr)
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
print(f"{len(X_tr)} clientes no treino | assinatura do treino {ASS} | KS conferido contra scipy")


def config(hp, features=(), over=True):
    return dict(hp=dict(hp), features=list(features), over=over)


def medir(cfg):
    def m():
        f = M.ajustar_prever_de(cfg["hp"], cfg["features"], cfg["over"])
        t0 = time.time()
        r = comum.medir_placar(X_tr, y_tr, f)
        r.update(arvores=float(np.median(f.arvores)), tempo_s=time.time() - t0)
        return r
    return m


def garantir(rotulo, cfg, contra=None):
    linha, medida = comum.garantir_placar(PASTA, M.MODELO, rotulo, cfg, medir(cfg), y_tr, ASS, contra)
    return linha


def mostrar(linhas, campos):
    rows = []
    for r in linhas:
        row = dict(rotulo=r["rotulo"], ks_kfold=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                   mse_treino=r.get("mse_treino"), mse_val=r["mse_val"], ce_val=r["ce_val"], arvores=r.get("arvores"))
        for c in campos:
            v = r.get(c)
            row[c] = "" if not v else f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
        rows.append(row)
    display(pd.DataFrame(rows).round(4))

5282 clientes no treino | assinatura do treino 4c2839956f33bab5 | KS conferido contra scipy


## 1. Trial 0

Os padrões do XGBoost: profundidade 6, taxa 0,3, sem amostragem de linhas nem de colunas, com o oversampling do grupo.

In [4]:
T0 = (M.MODELO, "trial0")
CFG0 = config(M.PADRAO)
linha_t0 = garantir("trial0", CFG0)
mostrar([linha_t0], [])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores
0,trial0,0.5157,0.0057,0.519,0.839,0.0973,0.155,0.4648,19.0


## 2. Busca no Optuna

Cada trial vale o KS médio das 5 dobras oficiais (semente 42), que não são as do placar.
O corte acontece depois da segunda dobra, como nos outros modelos.
A perda é um dos hiperparâmetros: entropia cruzada ou MSE sobre a probabilidade, como pedem os slides.

In [5]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
t0 = time.time()
st = comum.estudo_placar(M.MODELO, N_TRIALS, M.sugerir, M.ajustar_prever_de, M.PADRAO, X_tr, y_tr, pasta=PASTA_ESTUDO)
ok = [t for t in st.trials if t.state.name == "COMPLETE"]
print(f"{len(ok)} completos, {len(st.trials) - len(ok)} podados | {(time.time() - t0) / 60:.1f} min nesta execução")
print(f"KS trial 0 = {st.trials[0].value:.4f} -> melhor = {st.best_value:.4f} (trial {st.best_trial.number})")
print("melhor:", st.best_params)
resumo = comum.sumario_estudo(st)
print("por perda:", json.dumps(resumo["por_perda"], indent=1))
print("importâncias:", {k: round(v, 3) for k, v in resumo["importancias"].items()})
df = st.trials_dataframe()
display(df[df.state == "COMPLETE"].sort_values("value", ascending=False).head(10)[["number", "value"] + [c for c in df if c.startswith("params_")]].round(4))

83 completos, 20 podados | 54.3 min nesta execução
KS trial 0 = 0.5155 -> melhor = 0.5525 (trial 92)
melhor: {'max_depth': 1, 'min_child_weight': 12.016309392100725, 'learning_rate': 0.006495317996296761, 'subsample': 0.8027967850944152, 'colsample_bytree': 0.3184036466596406, 'reg_lambda': 7.969709180425087, 'reg_alpha': 0.0024326828168849215, 'gamma': 0.873591761035371, 'perda': 'ce'}


por perda: {
 "ce": {
  "n": 80,
  "melhor": 0.5525100498430285,
  "mediana": 0.5499202826010618,
  "media_top5": 0.5520436427863877
 },
 "mse": {
  "n": 3,
  "melhor": 0.5443677377764035,
  "mediana": 0.5383807305069732,
  "media_top5": 0.5401852036261545
 }
}
importâncias: {'colsample_bytree': 0.269, 'reg_alpha': 0.252, 'learning_rate': 0.22, 'gamma': 0.072, 'subsample': 0.067, 'reg_lambda': 0.065, 'min_child_weight': 0.034, 'max_depth': 0.018, 'perda': 0.004}


,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_perda,params_reg_alpha,params_reg_lambda,params_subsample
92,92,0.5525,0.3184,0.8736,0.0065,1,12.0163,ce,0.0024,7.9697,0.8028
77,77,0.5522,0.3425,0.9766,0.0068,1,11.2895,ce,0.0015,8.5402,0.8416
88,88,0.5519,0.4168,1.7370,0.0067,1,7.6230,ce,0.0017,15.6924,0.8209
87,87,0.5518,0.3261,1.2240,0.0066,1,4.9414,ce,0.0026,5.3026,0.8004
51,51,0.5518,0.3419,1.9361,0.0146,1,4.2530,ce,0.0057,1.1404,0.8664
100,100,0.5517,0.3263,0.9919,0.0076,1,12.1202,ce,0.0022,16.2737,0.8702
54,54,0.5515,0.3912,1.6338,0.0141,1,3.3404,ce,0.0169,1.6233,0.8465
80,80,0.5515,0.3236,1.2354,0.0062,1,12.8463,ce,0.0016,38.8945,0.8845
47,47,0.5515,0.3559,1.1935,0.0069,1,6.2454,ce,0.0024,5.7949,0.8238
58,58,0.5515,0.3438,1.6325,0.0079,1,5.5552,ce,0.0028,5.1271,0.8323


## 3. O melhor trial no placar

O KS do melhor trial nas dobras oficiais é otimista, porque foi escolhido nelas.
Aqui ele é medido nas 15 divisões do placar, contra o trial 0.

In [6]:
HP_OPTUNA = M.sugerir(optuna.trial.FixedTrial(st.best_params))
linha_opt = garantir("optuna", config(HP_OPTUNA), {"vs_trial0": T0})
mostrar([linha_t0, linha_opt], ["vs_trial0"])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,vs_trial0
0,trial0,0.5157,0.0057,0.519,0.8390,0.0973,0.155,0.4648,19.0,
1,optuna,0.5471,0.0067,0.539,0.8471,0.1561,0.164,0.4887,2282.0,"ganhou +0.0200 [+0.0039, +0.0349]"


## 4. Dados: oversampling e features

As duas perguntas do `docs/GUIA_DADOS.md`, com os hiperparâmetros do melhor trial parados.
Na sondagem do Diego, com outro XGBoost, nenhuma das duas mudou o KS; o MSE melhorou sem oversampling.

In [7]:
TRES = ("cliente_novo", "delta_preco", "tenure_1")
OPT = (M.MODELO, "optuna")
variantes = {"sem_over": config(HP_OPTUNA, over=False), "tres_features": config(HP_OPTUNA, features=TRES)}
linhas_var = {rot: garantir(rot, cfg, {"vs_optuna": OPT, "vs_trial0": T0}) for rot, cfg in variantes.items()}
mostrar([linha_opt, *linhas_var.values()], ["vs_optuna", "vs_trial0"])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,vs_optuna,vs_trial0
0,optuna,0.5471,0.0067,0.5390,0.8471,0.1561,0.1640,0.4887,2282.0,,"ganhou +0.0200 [+0.0039, +0.0349]"
1,sem_over,0.5489,0.0066,0.5396,0.8485,0.1316,0.1338,0.4131,2910.0,"empatou +0.0006 [-0.0044, +0.0087]","ganhou +0.0206 [+0.0067, +0.0367]"
2,tres_features,0.5497,0.0061,0.5397,0.8454,0.1567,0.1639,0.4909,1956.0,"empatou +0.0007 [-0.0082, +0.0103]","ganhou +0.0207 [+0.0047, +0.0360]"


## 5. Final

Fica o melhor trial, a menos que uma variante tenha **ganhado** dele: empate não troca.
Esta é a linha `("xgboost", "final")` que entra como `vs_xgboost` nos outros modelos.

In [8]:
ganharam = {rot: l for rot, l in linhas_var.items() if (l.get("vs_optuna") or {}).get("veredito") == "ganhou"}
if ganharam:
    rot_final = max(ganharam, key=lambda r: ganharam[r]["ks_oof"])
    CFG_FINAL, motivo = variantes[rot_final], f"{rot_final} ganhou do melhor trial no placar"
else:
    rot_final, CFG_FINAL = "optuna", config(HP_OPTUNA)
    motivo = "melhor trial do Optuna; nenhuma variante de dados ganhou dele no placar"
linha_final = garantir("final", CFG_FINAL, {"vs_trial0": T0})
comum.adotar_placar(PASTA, M.MODELO, "final", motivo)
print("final =", rot_final, "|", motivo)
print(json.dumps(CFG_FINAL, indent=1))
mostrar([linha_final], ["vs_trial0"])

final = optuna | melhor trial do Optuna; nenhuma variante de dados ganhou dele no placar
{
 "hp": {
  "max_depth": 1,
  "min_child_weight": 12.016309392100725,
  "learning_rate": 0.006495317996296761,
  "subsample": 0.8027967850944152,
  "colsample_bytree": 0.3184036466596406,
  "reg_lambda": 7.969709180425087,
  "reg_alpha": 0.0024326828168849215,
  "gamma": 0.873591761035371,
  "perda": "ce"
 },
 "features": [],
 "over": true
}


,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,vs_trial0
0,final,0.5471,0.0067,0.539,0.8471,0.1561,0.164,0.4887,2282.0,"ganhou +0.0200 [+0.0039, +0.0349]"


## 6. Teste

Só com `REVELAR_TESTE = True`, uma vez, com a configuração final já congelada.
O limiar das métricas de classificação vem da OOF do placar, nunca do teste.

In [9]:
if REVELAR_TESTE:
    fin = M.final(CFG_FINAL["hp"], X_tr, y_tr, X_te, N_SEMENTES, CFG_FINAL["features"], CFG_FINAL["over"])
    comum.registrar_teste_placar(PASTA, M.MODELO, "final", y_te.values, fin["p_sementes"])
    oof = np.load(os.path.join(PASTA, linha_final["oof"])).mean(0)
    limiar = comum.limiar_ks(y_tr.values, oof)
    p = np.mean(fin["p_sementes"], 0)
    m = comum.metricas(y_te.values, p, limiar)
    m.update(ks_ic95=comum.bootstrap_ks(y_te.values, p), ks_por_semente=[comum.ks(y_te.values, np.asarray(x)) for x in fin["p_sementes"]],
             arvores=fin["arvores"])
    json.dump(dict(config=CFG_FINAL, teste=m, **fin, y_teste=[int(v) for v in y_te]), open(os.path.join(PASTA_ESTUDO, "teste.json"), "w"), indent=1)
    print(json.dumps({k: v for k, v in m.items() if k != "confusao"}, indent=1)); print("confusão:", m["confusao"])
else:
    print("Teste não revelado (REVELAR_TESTE = False).")

Teste não revelado (REVELAR_TESTE = False).


In [10]:
display(comum.tabela_placar(PASTA).query("modelo == 'xgboost'").round(4))

,modelo,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_val,mse_treino,vs_trial0,vs_xgboost,ks_teste,auc_teste,vs_referencia,vs_adotado,vs_optuna
42,xgboost,trial0,0.5157,0.0057,0.5190,0.8390,0.1550,0.0973,NaN,None,NaN,NaN,NaN,NaN,NaN
43,xgboost,optuna,0.5471,0.0067,0.5390,0.8471,0.1640,0.1561,"ganhou (+0.0200 [+0.0039, +0.0349])",None,NaN,NaN,NaN,NaN,NaN
44,xgboost,sem_over,0.5489,0.0066,0.5396,0.8485,0.1338,0.1316,"ganhou (+0.0206 [+0.0067, +0.0367])",None,NaN,NaN,NaN,NaN,"empatou (+0.0006 [-0.0044, +0.0087])"
45,xgboost,tres_features,0.5497,0.0061,0.5397,0.8454,0.1639,0.1567,"ganhou (+0.0207 [+0.0047, +0.0360])",None,NaN,NaN,NaN,NaN,"empatou (+0.0007 [-0.0082, +0.0103])"
46,xgboost,final,0.5471,0.0067,0.5390,0.8471,0.1640,0.1561,"ganhou (+0.0200 [+0.0039, +0.0349])",None,NaN,NaN,NaN,NaN,NaN
